# Analyse stellar fit results

This notebook loads the results of a stellar spectrum fit produced by
`starships.stellar_fit` and analyses them:
- Convergence check (dynesty traces)
- Corner plot of the posterior
- **Prior vs. posterior comparison** (corner plot overlay + separate figure)
- Model vs. data comparison on all orders
- Summary table of best-fit parameters

**Before using this notebook**, run the fit on Narval:
```bash
python -m starships.stellar_fit my_config.yaml
```
or equivalently:
```python
import starships.stellar_fit as sf
sf.setup_stellar_fit('my_config.yaml')
sf.run_minimize()
sf.run_dynesty(save_file='dynesty_results.pkl')
```

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import corner
import dynesty.plotting as dyplot

import starships.stellar_fit as sf

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

---
## 1. Load config and results

In [ ]:
# --- Edit these two paths ---
CONFIG_FILE = 'my_stellar_fit_config.yaml'
RESULTS_FILE = 'dynesty_results.pkl'   # relative to walker_path in the config

# Load the config (this re-initialises all globals so we can call sf.* functions)
sf.setup_stellar_fit(CONFIG_FILE)

# Load the dynesty results
results_path = Path(sf.walker_path) / RESULTS_FILE
with open(results_path, 'rb') as f:
    dyn_results = pickle.load(f)

print(f"Loaded results from {results_path}")
print(f"Number of posterior samples: {len(dyn_results.samples)}")
print(f"log Z = {dyn_results.logz[-1]:.2f} ± {dyn_results.logzerr[-1]:.2f}")

---
## 2. Dynesty traces (convergence check)

The trace plot shows how the posterior samples were collected. Each panel
shows one parameter vs. the iteration number. Good convergence looks like
the samples exploring a stable region (not drifting systematically).

In [ ]:
param_labels = list(sf.params_prior.keys())

fig, _ = dyplot.traceplot(dyn_results, labels=param_labels, figsize=(10, 2 * len(param_labels)))
plt.tight_layout()
plt.show()

---
## 3. Corner plot

The corner plot shows the 1D and 2D marginal posterior distributions.
Look for:
- Unimodal distributions (single peak per parameter).
- Correlations between parameters (e.g. Teff–vsini can be degenerate).
- Posteriors that are not hitting the prior boundaries.

In [ ]:
from dynesty import utils as dyfunc

# Resample to get equally-weighted posterior samples
samples_eq, weights_eq = dyn_results.samples, np.exp(dyn_results.logwt - dyn_results.logz[-1])
samples_resampled = dyfunc.resample_equal(dyn_results.samples, weights_eq)

print(f"Number of equally-weighted samples: {len(samples_resampled)}")

In [ ]:
fig = corner.corner(
    samples_resampled,
    labels=param_labels,
    quantiles=[0.16, 0.50, 0.84],
    show_titles=True,
    title_kwargs={'fontsize': 10},
)
plt.show()

---
## 3bis. Prior vs. posterior visualization

Comparing the prior and posterior is essential to judge how much the data actually
constrain each parameter:
- A posterior much **narrower** than the prior means the data are informative.
- A posterior that **matches** the prior means the data have little leverage on that parameter.
- A posterior that **peaks away** from the prior mode can reveal tension with literature values.

Two views are provided below:
1. **Corner plot with prior overlay** — dashed red curves on the 1D diagonal panels.
2. **Separate figure** — one subplot per parameter, more readable when the prior range is much wider than the posterior.

In [ ]:
def compute_prior_pdf(x_arr, prior_info):
    """Compute the normalized prior PDF at the given sample points.

    Handles all prior types defined in stellar_fit: uniform, log_uniform,
    gaussian, split_gaussian, and combined_split_gaussian.

    Parameters
    ----------
    x_arr : array-like
        Points at which to evaluate the prior.
    prior_info : list
        Prior specification as stored in sf.params_prior:
        [prior_type, arg1, arg2, ...].

    Returns
    -------
    pdf : np.ndarray
        Prior PDF values at x_arr, normalized so that integral(pdf dx) ≈ 1.
    """
    prior_type = prior_info[0]
    args = prior_info[1:]
    x = np.asarray(x_arr, dtype=float)

    if prior_type == 'uniform':
        lo, hi = float(args[0]), float(args[1])
        pdf = np.where((x >= lo) & (x <= hi), 1.0, 0.0)

    elif prior_type == 'log_uniform':
        # Uniform in log-space: p(x) ∝ 1/x
        lo, hi = float(args[0]), float(args[1])
        pdf = np.where((x > 0) & (x >= lo) & (x <= hi), 1.0 / x, 0.0)

    elif prior_type == 'gaussian':
        mu, sigma = float(args[0]), float(args[1])
        pdf = np.exp(-0.5 * ((x - mu) / sigma) ** 2)

    elif prior_type == 'split_gaussian':
        # Asymmetric Gaussian: different sigma below and above the mode
        mu, sigma_lo, sigma_hi = float(args[0]), float(args[1]), float(args[2])
        sigmas = np.where(x < mu, sigma_lo, sigma_hi)
        pdf = np.exp(-0.5 * ((x - mu) / sigmas) ** 2)

    elif prior_type == 'combined_split_gaussian':
        # Product of independent split-Gaussian priors (one per literature reference)
        n_refs = len(args) // 3
        log_pdf = np.zeros_like(x, dtype=float)
        for i_ref in range(n_refs):
            mu       = float(args[3 * i_ref])
            sigma_lo = float(args[3 * i_ref + 1])
            sigma_hi = float(args[3 * i_ref + 2])
            sigmas = np.where(x < mu, sigma_lo, sigma_hi)
            log_pdf += -0.5 * ((x - mu) / sigmas) ** 2
        # Subtract the max before exp to avoid numerical underflow
        pdf = np.exp(log_pdf - log_pdf.max())

    else:
        # Unknown prior type — return a flat (uninformative) curve
        pdf = np.ones_like(x)

    # Normalize so the curve integrates to 1 (matches a density histogram)
    norm = np.trapz(pdf, x)
    if norm > 0:
        pdf = pdf / norm
    return pdf


def _prior_x_range(prior_info, samples_i, n_sigma=5):
    """Return a sensible x range that covers both the prior and the posterior.

    For Gaussian-type priors, extends n_sigma sigma away from each reference
    value. For uniform priors, uses the prior bounds. Always includes the
    full posterior extent as well.
    """
    prior_type = prior_info[0]
    args = prior_info[1:]

    if prior_type in ('uniform', 'log_uniform'):
        pr_lo, pr_hi = float(args[0]), float(args[1])
    elif prior_type == 'gaussian':
        mu, sigma = float(args[0]), float(args[1])
        pr_lo, pr_hi = mu - n_sigma * sigma, mu + n_sigma * sigma
    elif prior_type == 'split_gaussian':
        mu, sigma_lo, sigma_hi = float(args[0]), float(args[1]), float(args[2])
        pr_lo, pr_hi = mu - n_sigma * sigma_lo, mu + n_sigma * sigma_hi
    elif prior_type == 'combined_split_gaussian':
        n_refs = len(args) // 3
        pr_lo = min(float(args[3*j]) - n_sigma * float(args[3*j+1]) for j in range(n_refs))
        pr_hi = max(float(args[3*j]) + n_sigma * float(args[3*j+2]) for j in range(n_refs))
    else:
        pr_lo, pr_hi = samples_i.min(), samples_i.max()

    return min(pr_lo, samples_i.min()), max(pr_hi, samples_i.max())

In [ ]:
# ── View 1: corner plot with prior curves overlaid on the 1D diagonal panels ──
#
# corner.corner() arranges its axes in an N×N grid.
# The 1D marginal histograms sit on the diagonal (axes[i, i]).
# We draw the prior PDF (dashed red) rescaled to the histogram's y-axis range
# so it is visible without dominating the plot.

n_params = len(param_labels)

fig_prior = corner.corner(
    samples_resampled,
    labels=param_labels,
    quantiles=[0.16, 0.50, 0.84],
    show_titles=True,
    title_kwargs={'fontsize': 10},
)

# Retrieve the N×N grid of axes
corner_axes = np.array(fig_prior.axes).reshape((n_params, n_params))

for i, (key, prior_info) in enumerate(sf.params_prior.items()):
    ax = corner_axes[i, i]
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Evaluate the prior over the x range currently shown in the panel
    x_fine = np.linspace(xlim[0], xlim[1], 400)
    prior_pdf = compute_prior_pdf(x_fine, prior_info)

    if prior_pdf.max() > 0:
        # Rescale so the prior peak reaches 90% of the histogram's y range
        prior_scaled = prior_pdf / prior_pdf.max() * ylim[1] * 0.90
        ax.plot(x_fine, prior_scaled, 'r--', lw=1.5, alpha=0.85,
                label='Prior' if i == 0 else None)

# Add a legend only on the first (top-left) diagonal panel
corner_axes[0, 0].legend(fontsize=8, loc='upper left')

plt.suptitle('Posterior corner plot with prior overlay (dashed red)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── View 2: separate prior vs. posterior figure ────────────────────────────────
#
# One subplot per free parameter.  The x range is chosen to show the full prior
# support (or ±5σ for Gaussian-type priors) as well as the posterior extent.
# This view is most useful when the prior is much wider than the posterior —
# the corner plot view above clips the prior curve to the histogram's x range.

n_params = len(param_labels)

# Arrange subplots in a grid: 3 columns, as many rows as needed
n_cols = min(n_params, 3)
n_rows = (n_params + n_cols - 1) // n_cols
fig_sep, axes_sep = plt.subplots(
    n_rows, n_cols,
    figsize=(4.5 * n_cols, 3.5 * n_rows),
    constrained_layout=True,
)
# Flatten axes array for easy iteration, even if it's 1D
axes_flat = np.array(axes_sep).flatten()

for i, (key, prior_info) in enumerate(sf.params_prior.items()):
    ax = axes_flat[i]
    samples_i = samples_resampled[:, i]

    # Determine x range: covers prior support AND the full posterior extent
    x_lo, x_hi = _prior_x_range(prior_info, samples_i)
    x_range = np.linspace(x_lo, x_hi, 600)

    # Posterior histogram (density-normalized so it integrates to 1)
    ax.hist(samples_i, bins=40, density=True,
            color='steelblue', alpha=0.55, label='Posterior')

    # Prior PDF
    prior_pdf = compute_prior_pdf(x_range, prior_info)
    ax.plot(x_range, prior_pdf, 'r--', lw=1.8, alpha=0.9, label='Prior')

    # Posterior median and 68% credible interval
    q16, q50, q84 = np.percentile(samples_i, [16, 50, 84])
    ax.axvline(q50, color='navy', lw=1.2, ls='-')
    ax.axvspan(q16, q84, alpha=0.18, color='navy')
    ax.set_title(
        f'{key}\n{q50:.4g}  +{q84 - q50:.3g} / −{q50 - q16:.3g}',
        fontsize=9,
    )
    ax.set_xlabel(key, fontsize=10)
    ax.set_ylabel('Density' if i % n_cols == 0 else '', fontsize=9)
    ax.tick_params(labelsize=8)

    # Annotate the prior type in light gray at the top of the panel
    ax.text(0.97, 0.97, prior_info[0].replace('_', '\n'), transform=ax.transAxes,
            ha='right', va='top', fontsize=7, color='gray')

# Add a shared legend using the first subplot's handles
handles, labels_leg = axes_flat[0].get_legend_handles_labels()
fig_sep.legend(handles, labels_leg, loc='lower right', fontsize=9,
               bbox_to_anchor=(1.0, 0.0))

# Hide any unused subplot panels (if n_params is not a multiple of n_cols)
for j in range(n_params, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig_sep.suptitle('Prior (dashed red) vs. posterior (blue) — all free parameters',
                 fontsize=12)
plt.show()

---
## 4. Best-fit parameter summary

In [ ]:
# Compute median and 1-sigma intervals
q_lo, q_med, q_hi = np.percentile(samples_resampled, [16, 50, 84], axis=0)

print("Best-fit stellar parameters (median ± 1σ):")
print("-" * 50)
for i, key in enumerate(param_labels):
    err_lo = q_med[i] - q_lo[i]
    err_hi = q_hi[i] - q_med[i]
    print(f"  {key:12s} = {q_med[i]:9.3f}  +{err_hi:.3f} / -{err_lo:.3f}")

# Build the median theta_dict
theta_median = q_med
theta_dict_median = sf.unpack_theta(theta_median)

---
## 5. Model vs. data — all orders

`make_model_with_best_poly()` generates, for each order:
1. The PHOENIX + rotation model at the given parameters.
2. The optimal polynomial correction (solved analytically).
3. The corrected model = model × polynomial, which is directly comparable to the data.

In [ ]:
# Generate corrected models for the median parameter values
models_corrected, poly_corrections = sf.make_model_with_best_poly(
    theta_dict_median, return_poly=True
)

In [ ]:
# Plot all orders
n_orders = sf.ref_wave.shape[0]
fig, axes = plt.subplots(n_orders, 1, figsize=(12, 2.0 * n_orders))

for idx, ax in enumerate(axes):
    wv = sf.ref_wave[idx]
    data = sf.ref_spectrum[idx]
    uncert = sf.ref_uncert[idx]
    model_c = models_corrected[idx]

    ax.plot(wv, data, 'k', lw=0.5, label='Data' if idx == 0 else None)
    ax.fill_between(wv, data - uncert, data + uncert,
                    alpha=0.2, color='gray')
    ax.plot(wv, model_c, 'r', lw=0.5, alpha=0.8,
            label='Model + poly' if idx == 0 else None)
    ax.set_ylabel(f'#{idx}', fontsize=7)
    ax.set_ylim(0, 1.4)
    ax.tick_params(labelsize=7)

axes[0].legend(loc='upper right', fontsize=9)
axes[-1].set_xlabel('Wavelength (µm)')
plt.suptitle('Model vs. data — all orders (median parameters)', y=1.001)
plt.tight_layout()
plt.show()

In [ ]:
# Plot the residuals (data - model) for each order
fig, axes = plt.subplots(n_orders, 1, figsize=(12, 1.5 * n_orders))

for idx, ax in enumerate(axes):
    wv = sf.ref_wave[idx]
    residual = sf.ref_spectrum[idx] - models_corrected[idx]
    uncert = sf.ref_uncert[idx]

    ax.plot(wv, residual / uncert, 'k', lw=0.4)   # residual in units of sigma
    ax.axhline(0, color='r', linestyle='--', lw=0.8, alpha=0.5)
    ax.set_ylim(-5, 5)
    ax.set_ylabel(f'#{idx}', fontsize=7)
    ax.tick_params(labelsize=7)

axes[-1].set_xlabel('Wavelength (µm)')
plt.suptitle('Residuals in units of σ — all orders', y=1.001)
plt.tight_layout()
plt.show()

---
## 6. Polynomial corrections per order

Here we visualise the polynomial correction that was applied to each order.
A flat correction (close to 1 everywhere) means the normalisation was good.
A strongly varying correction indicates blaze residuals or normalisation issues.

In [ ]:
fig, axes = plt.subplots(n_orders, 1, figsize=(12, 1.5 * n_orders))

for idx, ax in enumerate(axes):
    wv = sf.ref_wave[idx]
    poly = poly_corrections[idx]

    ax.plot(wv, poly, 'b', lw=0.6)
    ax.axhline(1.0, color='k', linestyle='--', lw=0.6, alpha=0.4)
    ax.set_ylabel(f'#{idx}', fontsize=7)
    ax.tick_params(labelsize=7)

axes[-1].set_xlabel('Wavelength (µm)')
plt.suptitle('Per-order polynomial corrections exp(Φc*)', y=1.001)
plt.tight_layout()
plt.show()

---
## 7. Model uncertainty: draw from the posterior

To visualise how uncertain the model is, we generate the corrected model
for a sample of posterior draws and overplot them.

In [ ]:
# Draw a random subset of posterior samples
n_draws = 50
rng = np.random.default_rng(seed=42)
draw_indices = rng.integers(0, len(samples_resampled), size=n_draws)

# Choose a representative order to show
i_ord_show = 10
wv = sf.ref_wave[i_ord_show]

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(wv, sf.ref_spectrum[i_ord_show], 'k', lw=0.7, zorder=10, label='Data')

for idx_draw in draw_indices:
    theta_draw = samples_resampled[idx_draw]
    theta_dict_draw = sf.unpack_theta(theta_draw)
    models_draw = sf.make_model_with_best_poly(theta_dict_draw)
    ax.plot(wv, models_draw[i_ord_show], 'r', lw=0.4, alpha=0.15)

# Overplot the median model
ax.plot(wv, models_corrected[i_ord_show], 'r', lw=1.0, label='Median model')

ax.set_xlabel('Wavelength (µm)')
ax.set_ylabel('Normalised flux')
ax.set_title(f'Order #{i_ord_show} — posterior model uncertainty ({n_draws} draws)')
ax.legend()
plt.tight_layout()
plt.show()